# CoT-Style Multi-Round Planning with SecureRAG + Ollama (qwen3:0.6b)

This notebook demonstrates retrieval planning across multiple rounds, then answer synthesis with SecureRAG using a local Ollama model.

## What You Will Do

1. Launch a local SecureRAG simulation backend.
2. Build a corpus with overlapping evidence.
3. Use local Ollama `qwen3:0.6b` as planner/generator.
4. Inspect planner decisions across rounds (CoT-like retrieval refinement).
5. Run the full SecureRAG agent and inspect rounds, context size, and budget.

In [4]:
import atexit
import socket
import subprocess
import sys
import time
from pathlib import Path

import httpx

def _ensure_repo_on_path() -> Path:
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        if (base / "securerag" / "__init__.py").exists():
            base_str = str(base)
            if base_str not in sys.path:
                sys.path.insert(0, base_str)
            return base
    raise RuntimeError(
        "Could not locate the SecureRAG repository root. Open this notebook from the secure-rag workspace."
    )

REPO_ROOT = _ensure_repo_on_path()
print("Using repository root:", REPO_ROOT)

from securerag import PrivacyConfig, PrivacyProtocol, SecureRAGAgent
from securerag.corpus import CorpusBuilder
from securerag.llm import OllamaLLM
from securerag.models import RawDocument

Using repository root: /Users/sonnguyen/research/secure-rag


In [ ]:
def _free_port() -> int:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("127.0.0.1", 0))
        return int(s.getsockname()[1])


def _wait_for_health(base_url: str, proc: subprocess.Popen, timeout_s: float = 10.0) -> None:
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        if proc.poll() is not None:
            raise RuntimeError(
                "sim_server exited early. Ensure uvicorn is installed and the repository root is detected correctly."
            )
        try:
            r = httpx.get(f"{base_url}/health", timeout=0.5)
            if r.status_code == 200:
                return
        except Exception:
            pass
        time.sleep(0.1)
    raise RuntimeError(f"sim_server did not become healthy at {base_url}")


PORT = _free_port()
BASE_URL = f"http://127.0.0.1:{PORT}"
_env = dict(**__import__("os").environ)
_existing_pp = _env.get("PYTHONPATH", "")
_env["PYTHONPATH"] = str(REPO_ROOT) if not _existing_pp else f"{str(REPO_ROOT)}:{_existing_pp}"

SIM_PROC = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "uvicorn",
        "securerag.sim_server:app",
        "--host",
        "127.0.0.1",
        "--port",
        str(PORT),
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    env=_env,
    cwd=str(REPO_ROOT),
)
_wait_for_health(BASE_URL, SIM_PROC)
print("SecureRAG sim backend is up at", BASE_URL)


def _cleanup() -> None:
    if SIM_PROC.poll() is None:
        SIM_PROC.terminate()


_ = atexit.register(_cleanup)

SecureRAG sim backend is up at http://127.0.0.1:49988


<function __main__._cleanup() -> None>

In [ ]:
docs = [
    RawDocument(doc_id="audit", text="Q3 audit found delayed remediation for identity and access controls."),
    RawDocument(doc_id="ops", text="Operations incident logs show queue saturation and timeout spikes in July and August."),
    RawDocument(doc_id="vendor", text="Third-party vendor concentration increased after two contract renewals."),
    RawDocument(doc_id="policy", text="Security policy requires owner assignment, exception tracking, and quarterly closure KPIs."),
    RawDocument(doc_id="risk", text="Risk committee minutes mention compounding effects between vendor dependency and incident backlog."),
]

corpus = (
    CorpusBuilder(PrivacyProtocol.DIFF_PRIVACY, backend_url=BASE_URL)
    .with_chunk_size(160)
    .with_overlap(30)
    .add_documents(docs)
    .build()
)

cfg = PrivacyConfig(
    protocol=PrivacyProtocol.DIFF_PRIVACY,
    backend=BASE_URL,
    epsilon=200.0,
    noise_std=0.25,
    top_k=3,
    max_rounds=5,
)

llm = OllamaLLM(model="qwen3:0.6b", use_ollama=True)
agent = SecureRAGAgent.from_config(cfg, corpus=corpus, llm=llm)

In [ ]:
query = "Given Q3 evidence, what chain of causes best explains both rising incidents and elevated vendor risk, and what should leadership prioritize first?"

# CoT-style planner inspection across rounds.
d0 = llm.decide(query=query, context=[], round=0)
print("Round 0 decision:", d0)

seed_context = agent.retriever.retrieve(query, round_n=0)
d1 = llm.decide(query=query, context=seed_context, round=1)
print("Round 1 decision:", d1)

d2 = llm.decide(query=query, context=seed_context, round=2)
print("Round 2 decision:", d2)

In [ ]:
result = agent.run(query)
print("Answer:
", result.answer)
print()
print("Rounds used:", result.rounds)
print("Context size:", result.context_size)
print("Budget snapshot:", agent.budget_snapshot())

In [ ]:
if SIM_PROC.poll() is None:
    SIM_PROC.terminate()
print("Stopped local sim backend")